# GSM8K evaluation on Colab — Qwen 2.5 14B, 4 conditions

Runs **GSM8K-only** evaluation with:
- **Model:** Qwen/Qwen2.5-14B-Instruct
- **Conditions:** direct, short, medium, long
- **max_samples:** 100 (test split)
- **temperature:** 0

**Note:** 14B in bfloat16 needs ~28GB GPU memory. Use **Runtime → Change runtime type → GPU**. For free-tier T4, use `Qwen/Qwen2.5-7B-Instruct` in the config cell below.

In [ ]:
# Install dependencies (run once)
!pip install -q torch transformers datasets accelerate

In [ ]:
# Config
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct" for smaller GPU
SPLIT = "test"
MAX_SAMPLES = 100
SEED = 42
TEMPERATURE = 0.0
DTYPE = "bfloat16"
RESULTS_DIR = "results_large_gsm8k"

CONDITIONS = ["direct", "short", "medium", "long"]
COND_CFG = {
    "direct": {"steps": 0, "max_new_tokens": 64},
    "short":  {"steps": 2, "max_new_tokens": 128},
    "medium": {"steps": 4, "max_new_tokens": 256},
    "long":   {"steps": 8, "max_new_tokens": 512},
}

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results -> {RESULTS_DIR}/")

In [ ]:
# Answer extraction and scoring
import re
from typing import Optional

_FINAL_ANSWER_RE = re.compile(r"Final Answer\s*:\s*([-\+]?\$?\s*\d[\d,]*\.?\d*)", flags=re.IGNORECASE)
_LAST_NUMBER_RE = re.compile(r"([-\+]?\d[\d,]*\.?\d*)")

def _normalize_num(s: str) -> str:
    s = s.strip().replace("$", "").replace(",", "").rstrip(".")
    return s

def extract_number(text: str) -> Optional[str]:
    if not text:
        return None
    m = _FINAL_ANSWER_RE.search(text)
    if m:
        return _normalize_num(m.group(1))
    nums = _LAST_NUMBER_RE.findall(text)
    return _normalize_num(nums[-1]) if nums else None

def correct(pred: Optional[str], gold: Optional[str]) -> bool:
    if pred is None or gold is None:
        return False
    try:
        return float(pred) == float(gold)
    except Exception:
        return pred == gold

def estimate_steps(output_text: str) -> int:
    return len(re.findall(r"\bStep\s+\d+\s*:", output_text)) if output_text else 0

print("Scoring functions defined.")

In [ ]:
# Dataset and prompts
import random
import json
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

from datasets import load_dataset

SYSTEM_PROMPT = "You are a careful math solver. Follow formatting rules exactly."
USER_PROMPT_DIRECT = """Solve the following math problem.

Rules:
- Do NOT show any reasoning.
- Output exactly ONE line in the format:
  Final Answer: <number>
- Do not output anything else.

Problem:
{question}
"""
USER_PROMPT_COT = """Solve the following math problem.

Rules:
- Write EXACTLY {n_steps} reasoning steps.
- Each step MUST be a single line and MUST start with "Step k:" (k = 1..{n_steps}).
- Each step MUST be one short sentence (max 15 words).
- After the last step, output exactly ONE final line:
  Final Answer: <number>
- Do not output anything else.

Problem:
{question}
"""

def load_gsm8k(split: str) -> List[Dict[str, Any]]:
    ds = load_dataset("openai/gsm8k", "main", split=split)
    return [dict(x) for x in ds]

def pick_indices(n: int, max_samples: Optional[int], seed: int, shuffle: bool) -> List[int]:
    idx = list(range(n))
    if shuffle:
        random.Random(seed).shuffle(idx)
    if max_samples is not None and max_samples > 0:
        idx = idx[:max_samples]
    return idx

def build_user_prompt(question: str, condition: str, n_steps: int) -> str:
    if condition == "direct":
        return USER_PROMPT_DIRECT.format(question=question)
    return USER_PROMPT_COT.format(question=question, n_steps=n_steps)

def format_chat_if_possible(tokenizer, system_prompt: str, user_prompt: str, use_chat_template: bool = True) -> str:
    if use_chat_template and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return f"{system_prompt}\n\n{user_prompt}".strip()

rows = load_gsm8k(SPLIT)
idxs = pick_indices(len(rows), MAX_SAMPLES, SEED, shuffle=True)
print(f"Loaded GSM8K {SPLIT}: {len(rows)} total, running on {len(idxs)} samples.")

In [ ]:
# Load model once (shared across all 4 conditions)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

@dataclass
class RuntimeCfg:
    temperature: float
    top_p: float
    max_new_tokens: int
    use_chat_template: bool

def load_model(model_name: str, dtype: str):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token_id = tok.eos_token_id
    torch_dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}.get(dtype.lower())
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    model.eval()
    return model, tok

def generate(model, tokenizer, prompt: str, cfg: RuntimeCfg) -> Tuple[str, int, int]:
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, padding=False)
    input_ids = enc["input_ids"].to(model.device)
    attn = enc.get("attention_mask")
    if attn is not None:
        attn = attn.to(model.device)
    input_tokens = int(input_ids.shape[-1])
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn,
            do_sample=(cfg.temperature > 0),
            temperature=cfg.temperature if cfg.temperature > 0 else None,
            top_p=cfg.top_p,
            max_new_tokens=cfg.max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0, input_tokens:]
    output_tokens = int(gen_ids.shape[-1])
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text, input_tokens, output_tokens

def model_tag(model_name: str) -> str:
    return model_name.split("/")[-1].lower().replace(" ", "_")

print("Loading model (this may take a few minutes)...")
model, tokenizer = load_model(MODEL_NAME, DTYPE)
print("Model loaded.")

In [ ]:
# Run evaluation for all 4 conditions and save to RESULTS_DIR
tag = model_tag(MODEL_NAME)
for condition in CONDITIONS:
    n_steps = COND_CFG[condition]["steps"]
    max_new_tokens = COND_CFG[condition]["max_new_tokens"]
    out_name = f"{tag}_{condition}_{SPLIT}_n{MAX_SAMPLES}.jsonl"
    out_path = os.path.join(RESULTS_DIR, out_name)
    run_cfg = RuntimeCfg(temperature=TEMPERATURE, top_p=1.0, max_new_tokens=max_new_tokens, use_chat_template=True)
    correct_cnt = 0
    t0 = time.time()
    with open(out_path, "w", encoding="utf-8") as f:
        for k, i in enumerate(idxs, start=1):
            q = rows[i]["question"]
            gold_raw = rows[i]["answer"]
            gold_num = extract_number(gold_raw)
            user_prompt = build_user_prompt(q, condition, n_steps=n_steps)
            prompt = format_chat_if_possible(tokenizer, SYSTEM_PROMPT, user_prompt, True)
            start = time.time()
            gen_text, in_tok, out_tok = generate(model, tokenizer, prompt, run_cfg)
            latency = time.time() - start
            pred_num = extract_number(gen_text)
            ok = correct(pred_num, gold_num)
            correct_cnt += int(ok)
            rec = {
                "dataset": "gsm8k", "split": SPLIT, "qid": i, "model_name": MODEL_NAME, "condition": condition,
                "steps_target": n_steps, "max_new_tokens": max_new_tokens,
                "question": q, "gold_raw": gold_raw, "gold_num": gold_num,
                "pred_raw": gen_text, "pred_num": pred_num, "correct": ok,
                "input_tokens": in_tok, "output_tokens": out_tok, "steps_est": estimate_steps(gen_text),
                "latency_sec": latency, "seed": SEED,
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            if k % 25 == 0 or k == len(idxs):
                print(f"[{condition}] {k}/{len(idxs)} acc={correct_cnt/k:.3f}")
    total = time.time() - t0
    print(f"[done] {condition} -> {out_path} acc={correct_cnt/len(idxs):.4f} time={total:.1f}s")
print("All 4 conditions finished. Outputs in", RESULTS_DIR)